# 03 - BERT Model Comparison

Detaillierter Vergleich der drei BERT-Modelle (mBERT, GBERT, HateBERT).

**Inhalte:**
- Cross-Validation Ergebnisse analysieren
- Per-Class Performance vergleichen
- Statistische Signifikanztests
- Learning Curves aus Experiment 3
- Preprocessing Ablation aus Experiment 4

In [1]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.config import METRICS_DIR, PLOTS_DIR, LABEL_NAMES
from src.visualize import plot_confusion_matrix, plot_metrics_per_class

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print('Setup complete!')

Setup complete!


## 1. CV-Ergebnisse laden

In [2]:
# Lade alle BERT CV-Ergebnisse
models = ['GBERT', 'mBERT', 'HateBERT']
results = {}

for model_name in models:
    result_path = METRICS_DIR / f'{model_name}_cv_results.json'
    if result_path.exists():
        with open(result_path, 'r') as f:
            results[model_name] = json.load(f)
        print(f'✅ {model_name}: Geladen')
    else:
        print(f'⚠️  {model_name}: Nicht gefunden')

# Als DataFrame
df_summary = pd.DataFrame({
    model: {
        'F1 (Macro)': f"{data['f1_macro_mean']:.4f} ± {data['f1_macro_std']:.4f}",
        'Accuracy': f"{data['accuracy_mean']:.4f} ± {data['accuracy_std']:.4f}",
        'F1 OTHER': f"{data.get('f1_OTHER_mean', 0):.4f}",
        'F1 OFFENSE': f"{data.get('f1_OFFENSE_mean', 0):.4f}",
    }
    for model, data in results.items()
}).T

print('\n=== BERT-Modelle Zusammenfassung ===')
print(df_summary)

✅ GBERT: Geladen
✅ mBERT: Geladen
✅ HateBERT: Geladen

=== BERT-Modelle Zusammenfassung ===
               F1 (Macro)         Accuracy F1 OTHER F1 OFFENSE
GBERT     0.7923 ± 0.0120  0.8210 ± 0.0090   0.8694     0.7153
mBERT     0.7684 ± 0.0155  0.7965 ± 0.0103   0.8489     0.6880
HateBERT  0.6724 ± 0.0058  0.7123 ± 0.0102   0.7857     0.5590


## 2. Statistische Signifikanztests

In [3]:
# Paired t-test zwischen Modellen
print('\n=== Paarweise t-Tests (F1-Scores über 5 Folds) ===\n')

model_pairs = [
    ('GBERT', 'mBERT'),
    ('GBERT', 'HateBERT'),
    ('mBERT', 'HateBERT'),
]

for model_a, model_b in model_pairs:
    if model_a in results and model_b in results:
        f1_a = results[model_a].get('f1_macro_values', [])
        f1_b = results[model_b].get('f1_macro_values', [])
        
        if len(f1_a) == len(f1_b) and len(f1_a) > 0:
            t_stat, p_value = stats.ttest_rel(f1_a, f1_b)
            mean_diff = np.mean(f1_a) - np.mean(f1_b)
            
            sig_marker = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'n.s.'
            
            print(f'{model_a} vs. {model_b}:')
            print(f'  Differenz: {mean_diff:+.4f}')
            print(f'  t-Statistik: {t_stat:.3f}')
            print(f'  p-Wert: {p_value:.4f} {sig_marker}')
            print()


=== Paarweise t-Tests (F1-Scores über 5 Folds) ===

GBERT vs. mBERT:
  Differenz: +0.0239
  t-Statistik: 2.818
  p-Wert: 0.0479 *

GBERT vs. HateBERT:
  Differenz: +0.1200
  t-Statistik: 20.380
  p-Wert: 0.0000 ***

mBERT vs. HateBERT:
  Differenz: +0.0961
  t-Statistik: 17.421
  p-Wert: 0.0001 ***



## 3. Per-Class Performance

In [4]:
# Per-Class F1 Vergleich
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, label_name in enumerate(LABEL_NAMES):
    ax = axes[i]
    
    model_names = []
    f1_means = []
    f1_stds = []
    
    for model in models:
        if model in results:
            model_names.append(model)
            f1_means.append(results[model].get(f'f1_{label_name}_mean', 0))
            f1_stds.append(results[model].get(f'f1_{label_name}_std', 0))
    
    colors = ['#2ecc71' if m == 'GBERT' else '#3498db' for m in model_names]
    bars = ax.bar(range(len(model_names)), f1_means, yerr=f1_stds, 
                  color=colors, capsize=5, edgecolor='gray')
    
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names)
    ax.set_ylabel('F1-Score', fontsize=12)
    ax.set_title(f'F1-Score für Klasse: {label_name}', fontsize=13)
    ax.set_ylim(0, 1)
    ax.grid(True, axis='y', alpha=0.3)
    
    # Werte über Balken
    for bar, mean in zip(bars, f1_means):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{mean:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'per_class_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Users\pauls\AppData\Local\Temp\ipykernel_12692\4046938932.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Learning Curve (Experiment 3)

In [5]:
# Lade Data Size Variation Ergebnisse
data_size_path = METRICS_DIR / 'data_size_variation_results.json'

if data_size_path.exists():
    with open(data_size_path, 'r') as f:
        data_size_results = json.load(f)
    
    # Parse results
    sizes = []
    f1_means = []
    f1_stds = []
    acc_means = []
    acc_stds = []
    
    for size_key in sorted(data_size_results.get("results", data_size_results).keys(), key=lambda x: float(x.strip('%'))):
        data = data_size_results.get("results", data_size_results)[size_key]
        sizes.append(int(size_key.strip('%')))
        f1_means.append(data['f1_macro_mean'])
        f1_stds.append(data['f1_macro_std'])
        acc_means.append(data['accuracy_mean'])
        acc_stds.append(data['accuracy_std'])
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.errorbar(sizes, f1_means, yerr=f1_stds, marker='o', linewidth=2, 
                markersize=8, capsize=5, label='F1-Score', color='#3498db')
    ax.errorbar(sizes, acc_means, yerr=acc_stds, marker='s', linewidth=2, 
                markersize=8, capsize=5, label='Accuracy', color='#e74c3c')
    
    ax.set_xlabel('Trainingsdatengröße (%)', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Learning Curve: GBERT auf GermEval 2018', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.5, 0.9)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'learning_curve.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print('\n=== Data Efficiency ===')
    for size, f1, acc in zip(sizes, f1_means, acc_means):
        print(f'{size:3d}% Daten: F1={f1:.4f}, Acc={acc:.4f}')
else:
    print('⚠️  Learning Curve Daten nicht gefunden (Experiment 3 noch nicht gelaufen)')


=== Data Efficiency ===
 10% Daten: F1=0.5214, Acc=0.5845
 25% Daten: F1=0.7097, Acc=0.7651
 50% Daten: F1=0.7880, Acc=0.8144
 75% Daten: F1=0.7914, Acc=0.8208
100% Daten: F1=0.8097, Acc=0.8334


C:\Users\pauls\AppData\Local\Temp\ipykernel_12692\1174904678.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Preprocessing Ablation (Experiment 4)

In [6]:
# Lade Preprocessing Ablation Ergebnisse
preprocessing_path = METRICS_DIR / 'preprocessing_ablation_results.json'

if preprocessing_path.exists():
    with open(preprocessing_path, 'r') as f:
        preprocessing_data = json.load(f)
    
    results_dict = preprocessing_data.get('results', {})
    
    # Als DataFrame
    prep_df = pd.DataFrame({
        variant: {
            'F1 (Macro)': data['f1_macro_mean'],
            'F1 Std': data['f1_macro_std'],
            'Accuracy': data['accuracy_mean'],
        }
        for variant, data in results_dict.items()
    }).T
    
    prep_df = prep_df.sort_values('F1 (Macro)', ascending=False)
    
    print('\n=== Preprocessing Ablation ===')
    print(prep_df)
    
    # Barplot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    variants = prep_df.index.tolist()
    f1_values = prep_df['F1 (Macro)'].values
    f1_stds = prep_df['F1 Std'].values
    
    colors = ['#2ecc71' if i == 0 else '#95a5a6' for i in range(len(variants))]
    bars = ax.barh(range(len(variants)), f1_values, xerr=f1_stds, 
                   color=colors, capsize=4, edgecolor='gray')
    
    ax.set_yticks(range(len(variants)))
    ax.set_yticklabels(variants)
    ax.invert_yaxis()
    ax.set_xlabel('F1-Score (Macro)', fontsize=12)
    ax.set_title('Preprocessing Ablation: GBERT', fontsize=14)
    ax.grid(True, axis='x', alpha=0.3)
    
    # Werte an Balken
    for i, (mean, std) in enumerate(zip(f1_values, f1_stds)):
        ax.text(mean + std + 0.005, i, f'{mean:.4f}', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'preprocessing_ablation.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  Preprocessing Ablation Daten nicht gefunden (Experiment 4 noch nicht gelaufen)')


=== Preprocessing Ablation ===
                              F1 (Macro)    F1 Std  Accuracy
full_preprocessing              0.809705  0.010649  0.833400
remove_urls                     0.805310  0.005881  0.829322
normalize_usernames             0.805015  0.005840  0.829934
original                        0.803577  0.007404  0.827487
remove_emojis                   0.803059  0.007840  0.825652
full_preprocessing_lowercase    0.792317  0.012029  0.820962
lowercase                       0.789058  0.011265  0.816475


C:\Users\pauls\AppData\Local\Temp\ipykernel_12692\2846783923.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Zusammenfassung für Paper

In [7]:
print('\n' + '='*70)
print('ZUSAMMENFASSUNG FÜR HAUSARBEIT')
print('='*70)

# Bestes Modell
best_model = max(results.items(), key=lambda x: x[1]['f1_macro_mean'])
print(f'\n✅ Bestes Modell: {best_model[0]}')
print(f'   F1 (Macro):  {best_model[1]["f1_macro_mean"]:.4f} ± {best_model[1]["f1_macro_std"]:.4f}')
print(f'   Accuracy:    {best_model[1]["accuracy_mean"]:.4f} ± {best_model[1]["accuracy_std"]:.4f}')

# Modell-Ranking
print('\n📊 Modell-Ranking (nach F1):')
for i, (model, data) in enumerate(sorted(results.items(), key=lambda x: x[1]['f1_macro_mean'], reverse=True), 1):
    print(f'   {i}. {model}: {data["f1_macro_mean"]:.4f}')

# Key Findings
print('\n🔍 Key Findings:')
if 'GBERT' in results and 'mBERT' in results:
    gbert_f1 = results['GBERT']['f1_macro_mean']
    mbert_f1 = results['mBERT']['f1_macro_mean']
    diff = gbert_f1 - mbert_f1
    print(f'   • GBERT übertrifft mBERT um {diff:.4f} F1-Punkte ({diff/mbert_f1*100:.1f}%)')

if 'GBERT' in results and 'HateBERT' in results:
    gbert_f1 = results['GBERT']['f1_macro_mean']
    hatebert_f1 = results['HateBERT']['f1_macro_mean']
    diff = gbert_f1 - hatebert_f1
    print(f'   • GBERT übertrifft HateBERT um {diff:.4f} F1-Punkte ({diff/hatebert_f1*100:.1f}%)')

print(f'   • Language-specific model (GBERT) schlägt multilingual (mBERT)')
print(f'   • Domain-specific falscher Sprache (HateBERT) schlechter als generisches GBERT')


ZUSAMMENFASSUNG FÜR HAUSARBEIT

✅ Bestes Modell: GBERT
   F1 (Macro):  0.7923 ± 0.0120
   Accuracy:    0.8210 ± 0.0090

📊 Modell-Ranking (nach F1):
   1. GBERT: 0.7923
   2. mBERT: 0.7684
   3. HateBERT: 0.6724

🔍 Key Findings:
   • GBERT übertrifft mBERT um 0.0239 F1-Punkte (3.1%)
   • GBERT übertrifft HateBERT um 0.1200 F1-Punkte (17.8%)
   • Language-specific model (GBERT) schlägt multilingual (mBERT)
   • Domain-specific falscher Sprache (HateBERT) schlechter als generisches GBERT
